In [1]:
### install required library 
!pip install torch


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
### install required library 
!pip install torch_geometric


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
### ### install required library 
import pandas as pd
from rdkit import Chem
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
import numpy as np
import networkx as nx

In [2]:
# Load datasets
train_df = pd.read_csv('../data/final_data/final_unique_train.csv')
test_df = pd.read_csv('../data/final_data/final_unique_test.csv')

In [3]:
# Function to convert SMILES to molecular graphs
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    G = nx.Graph()
    for atom in mol.GetAtoms():
        G.add_node(atom.GetIdx(),
                   atomic_num=atom.GetAtomicNum(),
                   atomic_mass=atom.GetMass(),
                   hybridization=str(atom.GetHybridization()),
                   formal_charge=atom.GetFormalCharge(),
                   num_hs=atom.GetTotalNumHs(),
                   aromaticity=atom.GetIsAromatic())

    bond_type_dict = {'SINGLE': 1, 'DOUBLE': 2, 'TRIPLE': 3, 'AROMATIC': 4}
    for bond in mol.GetBonds():
        G.add_edge(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), bond_type=bond_type_dict.get(str(bond.GetBondType()), 0))

    return G

train_graphs = [smiles_to_graph(smiles) for smiles in train_df['smiles_canon']]
test_graphs = [smiles_to_graph(smiles) for smiles in test_df['smiles_canon']]


[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:52] WARNING: not removing hydrogen atom without neighbors
[00:27:53] WARNING: not removing hydrogen atom without neighbors
[00:27:53] WARNING: not removing hydrogen atom without neighbors
[00:27:53] WARNING: not removing hydrogen atom without neighbors
[00:27:53] WARNING: not removing hydrogen atom without neighbors
[00:27:53] WARNING: not removing hydrogen atom without neighbors


In [4]:

# Function to create PyTorch Geometric Data objects with extended features
def create_data_list(graphs, df):
    data_list = []
    for i, graph in enumerate(graphs):
        if graph is None:
            continue
        atom_features = nx.get_node_attributes(graph, 'atomic_num')
        num_nodes = len(atom_features)
        x = np.zeros((num_nodes, 6), dtype=np.float32)  # Adjust dimension as per added features
        for node_id, attrs in graph.nodes(data=True):
            x[node_id, 0] = attrs['atomic_num']
            x[node_id, 1] = attrs['atomic_mass']
            x[node_id, 2] = {'SP': 0, 'SP2': 1, 'SP3': 2}.get(attrs['hybridization'], 0)
            x[node_id, 3] = attrs['formal_charge']
            x[node_id, 4] = attrs['num_hs']
            x[node_id, 5] = attrs['aromaticity']

        edge_index = np.array(list(graph.edges), dtype=np.int64).T
        edge_attr = np.array([graph[u][v]['bond_type'] for u, v in graph.edges], dtype=np.float32)

        y = np.array([df.loc[i, 'LogS']], dtype=np.float32)

        data_list.append(Data(x=torch.tensor(x, dtype=torch.float),
                              edge_index=torch.tensor(edge_index, dtype=torch.long),
                              edge_attr=torch.tensor(edge_attr, dtype=torch.float),
                              y=torch.tensor(y, dtype=torch.float)))

    return data_list

train_data_list = create_data_list(train_graphs, train_df)
test_data_list = create_data_list(test_graphs, test_df)

# Filter out None values
train_data_list = [data for data in train_data_list if data is not None]
test_data_list = [data for data in test_data_list if data is not None]

# Create DataLoader
train_loader = DataLoader(train_data_list, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data_list, batch_size=64, shuffle=False)

# Define a more complex MPNN model architecture
class MPNN(nn.Module):
    def __init__(self):
        super(MPNN, self).__init__()
        self.conv1 = GCNConv(6, 64)  # Adjust input dimension as per extended features
        self.conv2 = GCNConv(64, 128)
        self.conv3 = GCNConv(128, 64)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.conv3(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Train and evaluate the model
model = MPNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()


/opt/miniconda3/envs/env2/lib/python3.8/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [5]:
# Training loop
model.train()
for epoch in range(200):
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out.view(-1), batch.y.view(-1))
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')

Epoch 1, Loss: 4.925538539886475
Epoch 2, Loss: 3.279299259185791
Epoch 3, Loss: 3.59143328666687
Epoch 4, Loss: 4.091395854949951
Epoch 5, Loss: 3.710634231567383
Epoch 6, Loss: 1.9896740913391113
Epoch 7, Loss: 4.963743686676025
Epoch 8, Loss: 8.156213760375977
Epoch 9, Loss: 2.5258755683898926
Epoch 10, Loss: 7.923193454742432
Epoch 11, Loss: 3.497101068496704
Epoch 12, Loss: 3.5517938137054443
Epoch 13, Loss: 4.304886341094971
Epoch 14, Loss: 3.7501347064971924
Epoch 15, Loss: 5.348857879638672
Epoch 16, Loss: 6.862329006195068
Epoch 17, Loss: 3.0151219367980957
Epoch 18, Loss: 4.590122699737549
Epoch 19, Loss: 2.4060299396514893
Epoch 20, Loss: 3.8952016830444336
Epoch 21, Loss: 1.7952417135238647
Epoch 22, Loss: 1.5275377035140991
Epoch 23, Loss: 4.617563724517822
Epoch 24, Loss: 4.020464897155762
Epoch 25, Loss: 2.0255954265594482
Epoch 26, Loss: 1.7797671556472778
Epoch 27, Loss: 6.319666385650635
Epoch 28, Loss: 3.7895569801330566
Epoch 29, Loss: 3.3410613536834717
Epoch 30, L

In [6]:
# Evaluation loop
model.eval()
predictions = []
actuals = []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch)
        predictions.extend(out.view(-1).tolist())
        actuals.extend(batch.y.view(-1).tolist())

predictions = np.array(predictions)
actuals = np.array(actuals)

In [7]:
### Result of the model 
mse = mean_squared_error(actuals, predictions)
mae = mean_absolute_error(actuals, predictions)
rmse = np.sqrt(np.mean((predictions - actuals) ** 2))
r2 = r2_score(actuals, predictions)
print(f'MSE: {mse:.4f}, MAE: {mae:.4f}, R^2: {r2:.4f},RMSE: {rmse:.4f}')


MSE: 0.9996, MAE: 0.7638, R^2: 0.7603,RMSE: 0.9998


In [ ]:
### End here 